# Cross-run analyses of the hierarchy-design runs

Analyses that cut across the runs of `hierarchy_design_runs.ipynb` (one run per section there) live here, so that
notebook keeps its one-section-per-run structure. The setup cell loads the first-layer trajectory (every 10 epochs) of
every run from `results/designed_init/` through a light loader that reads only the weight stacks; add a run to `RUN_INDEX`
when a new one appears. Sections:

1. Where deviating probes go: lineage in the ODT tree (moved here from section 22 of the runs notebook on 2026-10-10).
2. Path signs in run 26: which of the 96 000 paths flip sign, and why (2026-10-12).


In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
%load_ext autoreload
%autoreload 2
from layer_lenses.designed_init import HIERARCHY_TARGETS, NODE_LEVEL, node_lineage, lineage_chance, probe_deviation_lineage
from layer_lenses.init_sweep import load_tree

RESULTS = Path("../results/designed_init")
tree = load_tree(RESULTS / "hierarchy_replayable" / "tree.npz")            # the same COB-ODT (tree seed 5178) in every run

# run number -> (folder, tag or None for a replay folder, first-layer width); the replay folders hold weights.npz written by save_designed_run
RUN_INDEX = {1: ("hierarchy_kaiming_cosine", "linear_free_kaiming_cosine_seed0", 80), 2: ("hierarchy_kaiming_cosine", "linear_frozen_kaiming_cosine_seed0", 80), 3: ("hierarchy_kaiming_cosine_big", "linear_frozen_kaiming_cosine_big_seed0", 62),
             4: ("hierarchy_kaiming_cosine_isolate", "linear_frozen_kaiming_cosine_small_62_seed0", 62), 5: ("hierarchy_kaiming_cosine_isolate", "linear_frozen_kaiming_cosine_big_80_seed0", 80), 6: ("hierarchy_kaiming_cosine_balanced", "linear_frozen_kaiming_cosine_bal_62_seed0", 62),
             7: ("hierarchy_kaiming_cosine_balanced", "linear_frozen_kaiming_cosine_bal_80_seed0", 80), 8: ("hierarchy_kaiming_cosine_pathsign", "linear_frozen_kaiming_cosine_pathsign_80_seed0", 80), 9: ("hierarchy_kaiming_cosine_cueeq", "linear_frozen_kaiming_cosine_cueeq_80_seed0", 80),
             10: ("hierarchy_kaiming_cosine_balpaths", "linear_frozen_kaiming_cosine_balpaths_80_seed0", 80), 11: ("hierarchy_replayable", None, 80), 12: ("hierarchy_replayable_probe001", None, 80), 13: ("hierarchy_branch750_sgd_frozenL1", None, 80),
             14: ("hierarchy_branch750_adam_frozenL1", None, 80), 15: ("hierarchy_branch750_adam_all", None, 80), 16: ("hierarchy_rotlock099", None, 80), 17: ("hierarchy_scalaradam", None, 80), 18: ("hierarchy_scalaradam_lr003", None, 80),
             19: ("hierarchy_scalaradam_lr01", None, 80), 20: ("hierarchy_scalaradam_lr03", None, 80), 21: ("hierarchy_replayable_62", None, 62), 22: ("hierarchy_idealL1", None, 62), 23: ("hierarchy_idealL1_zerocue", None, 62),
             24: ("hierarchy_replayable_probepos", None, 80), 25: ("hierarchy_replayable_w2neg", None, 80), 26: ("hierarchy_replayable_cuev", None, 80),
             27: ("hierarchy_symprobes_80", None, 80), 28: ("hierarchy_symprobes_62", None, 62), 29: ("hierarchy_symprobes_80_ds2", None, 80), 30: ("hierarchy_symprobes_62_ds2", None, 62),
             31: ("hierarchy_cuev_62", None, 62), 32: ("hierarchy_cuev_62_w3seed1", None, 62), 33: ("hierarchy_cuev_62_ds2", None, 62), 34: ("hierarchy_cuev_62_w3seed1_ds2", None, 62), 35: ("hierarchy_cuev_62_ps7", None, 62), 36: ("hierarchy_cuev_62_w3seed1_ps7", None, 62),
             37: ("hierarchy_equivariant_60", None, 80), 38: ("hierarchy_equivariant_60_nospare", None, 62),
             39: ("hierarchy_equivariant_60_nospare_cue01", None, 62), 40: ("hierarchy_equivariant_60_nospare_pgf", None, 62), 41: ("hierarchy_equivariant_60_nospare_l3x4", None, 62)}

def load_weight_stacks(n, keys=("hidden_layers.0.weight",)):
    """Weight stacks (epochs x shape) of run n for the given parameter names, plus the checkpoint epochs; None if the files are absent."""
    folder, tag, _ = RUN_INDEX[n]
    f = RESULTS / folder / ("runs/" + tag + ".npz" if tag else "replay/weights.npz")
    if not f.exists():
        return None
    z = np.load(f); return {"epochs": z["epochs"], "W": {k: z["w__" + k] for k in keys}}

RUNS = {n: r for n in RUN_INDEX if (r := load_weight_stacks(n)) is not None}
print("runs with a stored first-layer trajectory:", sorted(RUNS), "| missing:", sorted(set(RUN_INDEX) - set(RUNS)))

## 1. Where deviating probes go: lineage in the ODT tree, across runs 1-21

A probe that ends captured (|cos| >= 0.95) on a node other than its designed target is a *deviation*. `probe_deviation_lineage`
classifies the captured node relative to the target in the tree (heap indexing: children of node i are 2i+1 and 2i+2, the
split at i sends w_i . x > 0 to the right child 2i+2): `ancestor`, `descendant` or `unrelated`, with the level gap and the
*side*: for a descendant, whether it sits in the target's positive-side subtree; for an ancestor, on which side of the
ancestor the target lies. `chance` is the fraction of the other 30 nodes on the target's lineage (1 for the root, 1/2 for
level 1, 8/30 for level 2, 5/30 for level 3).

**Findings (end state of every run as of run 21; the cell below recomputes them for every run loaded above, run 22 included).**
- Runs 10-21, i.e. every run with the balanced +-10-quanta third-layer paths: 66 deviations, 35 to an ancestor and 31 to a
  descendant, **0 unrelated** (chance level 27%). The descendants are always a direct child (level gap 1); the ancestors are
  the parent in 33 cases and the root in 2 (run 12). The same holds at every checkpoint along the trajectories: no probe of
  runs 10-21 ever captured an unrelated node.
- Runs 1-9: 79 deviations, 61% lineage-related (chance 29%): 31 unrelated, 9 of them in run 2 and 7 in run 4. So the
  observation is a strong tendency under the earlier designs and an exact rule under the current one.
- A second regularity: a probe that climbs to an ancestor always takes the ancestor's pole on the side that contains its
  target (45 of 45 across all runs): -u_2's probe becomes +u_0 (node 2 is the positive child of the root), +u_1's becomes
  -u_0, +u_4's becomes +u_1, -u_5's becomes -u_2. The neuron ends as a coarser gate for the same region of input space.
- Descendants mostly land in the subtree on the probe's own pole side (25 of 31 in runs 10-21, 20 of 38 in runs 1-9): +u_7's
  probe goes to u_16 (the positive child), -u_14's to u_29 (the negative child), -u_7's to u_15, -u_9's to u_19.
- Who deviates in runs 10-21: rows 5 (11 runs), 14 (11), 29 (11), 8 (8), 2 (6), 17 (3); the rest once or twice. These are
  the same substitutes seen run after run, so the deviation is a property of the init and the data, not of the optimizer
  (SGD, AdamW, ScalarAdam, frozen or locked first layer all agree).


In [ ]:
ALL_RUNS = RUNS
rows = []
for n, rr in ALL_RUNS.items():
    t = probe_deviation_lineage(rr["W"]["hidden_layers.0.weight"][-1], tree); t.insert(0, "run", n); rows.append(t)
E = pd.concat(rows, ignore_index=True); dev = E[E.deviated].copy()
print(f"runs loaded: {sorted(ALL_RUNS)}; probes {len(E)}, deviations {len(dev)}, not captured {int((~E.captured).sum())}")
for lab, sel in (("all", dev.run > 0), ("runs 1-9", dev.run <= 9), ("runs 10-21", dev.run >= 10)):
    d = dev[sel]
    if len(d) == 0: continue
    a = d[d.relation == "ancestor"]; s = d[d.relation == "descendant"]
    print(f"{lab}: {len(d)} deviations: ancestor {len(a)}, descendant {len(s)}, unrelated {int((d.relation == 'unrelated').sum())} -> lineage-related {d.relation.isin(['ancestor', 'descendant']).mean() * 100:.0f}% (chance {d.chance.mean() * 100:.0f}%)"
          f"; ancestor pole == side containing the target: {int((np.sign(a.end_cos) == a.side.map({'+': 1, '-': -1})).sum())}/{len(a)}; descendant on the probe's own pole side: {int((s.pole == s.side.map({'+': 1, '-': -1})).sum())}/{len(s)}")
print("unrelated deviations per run:", dev.groupby("run").relation.apply(lambda r: int((r == "unrelated").sum())).to_dict())
display(dev[dev.run >= 10][["run", "probe", "target", "target_level", "pole", "end_node", "end_cos", "relation", "level_gap", "side"]].round(3).reset_index(drop=True))
# along the trajectories: every off-target capture at any checkpoint
traj = []
for n, rr in ALL_RUNS.items():
    W_ = rr["W"]["hidden_layers.0.weight"][:, :30]; C_ = np.abs(W_ / np.linalg.norm(W_, axis=2, keepdims=True) @ tree.w_list.T) >= 0.95
    for p_ in range(30):
        for j in np.nonzero(C_[:, p_].any(0))[0]:
            if j != p_ // 2: traj.append({"run": n, "probe": p_, "target": p_ // 2, "node": int(j), "relation": node_lineage(p_ // 2, int(j))["relation"], "first_epoch": int(rr["epochs"][np.argmax(C_[:, p_, j])])})
T = pd.DataFrame(traj); print("off-target captures at any checkpoint:", T.relation.value_counts().to_dict(), "| unrelated ones in runs:", sorted(T[T.relation == "unrelated"].run.unique().tolist()))

## 2. Path signs in run 26: which of the 80 x 30 x 40 paths flip sign

The logit is a sum over the 96 000 three-edge paths (row i, unit j, neuron k) of w_out[k] W3[k, j] W2[j, i] times the gates and the
row's projection. The output weights never flip in the balanced designs (`dL/dw_k = E[(p - y) h_k]` with the feature h_k built through
w_k itself, see the runs notebook). This section asks how much of that carries over to paths, on run 26 (the anchor run). Sign changes
are read from the checkpoints every 10 epochs, so a weight that crosses zero and comes back within 1000 steps is invisible.

**Findings.**
- The design is sparse: only 5720 of the 96 000 paths are nonzero at init (W2 has 286 nonzero entries, W3 600). Of these designed paths
  30% (1745) end with the opposite sign, so "paths keep their sign" is false as stated. The flips are structured, though, and come from
  three edge classes; the output weights (40) never flip.
- **W3 minority edges (80 of the 150).** Each unit has 20 one-quantum paths to the logit, 15 with its net sign and 5 against it. 80 of
  the 150 minority edges go through zero and end negative (the path then carries the unit's net sign); the other 70 shrink (median 0.2
  vs 0.75 for the majority edges). One majority edge flips (to -0.06). The 600 W3 edges born from zero take the unit's net sign in 557
  cases. So the per-unit fraction of signed path mass on the unit's net sign goes from 0.75 at init to 1.00 for 10 units and above
  0.94 for 23; it is 0.82-0.88 for units 11, 13, 19, 22, 25 (no minority flips) and 0.59 / 0.62 for units 27 and 29, whose probes failed
  (rows 27 and 29 end at cos 0.73 on u_16 and at cos -0.34) and whose born W3 edges take the *opposite* sign (1 of 20 on the init net
  sign): those two units were re-purposed with the other sign. The minority flips happen after the probes are captured (first flip at
  epoch 350, median 870, last 7940). This is the output-weight argument one layer
  down: `dL/dW3[k, j] = w_k E[(p - y) 1_k h_j]` with w_k of fixed sign, and h_j is built through its *net* path sign, so an edge whose
  path sign agrees with the unit's net sign is pushed away from zero like an output weight, and one that disagrees is pushed through it.
- **W2 probe edges (5 of 30).** Rows 11, 13, 15, 21, 25 (all negative-pole probes on odd units) go from -1 to +0.04..0.1 (+0.57 for
  unit 21, which also recruits a cue at +4.2). These are prunings that overshoot zero, not re-wirings: the unit stops reading its probe.
- **W2 cue edges (68 of 256).** 56 end below 0.6 x their init magnitude, most below 0.2 (pruned through zero). 12 re-grow with the
  opposite sign up to 8 x their init, and every one of them is a cue that left its level-4 normal for an internal node (its parent or
  grandparent: +u_16 -> +u_7, -u_29 -> +u_6, +u_17 -> -u_8, +u_21 -> -u_10, -u_26 -> +u_12): the row changed identity, so the unit
  re-chose the sign it reads it with. The lineage rule of section 1 shows up here as a sign flip.
- **Born paths** carry the end state: 90 280 paths are zero at init, 62 173 end above 10% of the designed paths' median magnitude, and
  they hold 82% of the end sum of |path products| (the flipped designed paths hold 3%). 61 972 of the significant born paths and 5626 of
  the 5720 designed paths keep one sign over the last 2000 epochs.
- The gradient-flow invariants (w_k^2 - |W3[k, :]|^2 per neuron, and the analogues per unit and row) are not what protects the output
  weights: they drift by up to 8 (neurons), 126 (units) and 46 (rows) under momentum SGD, and are negative at init anyway
  (|w_k| = 0.22 < |W3[k, :]| = 1.41). The protection is dynamical: a weight is pushed away from zero whenever the feature it reads was
  built with its own sign, and that is the case for every output weight, for the majority W3 edges and for the probe edges that keep
  their row.

**Reading.** The path-level version of "no sign flips" is a consensus statement, not a conservation statement: SGD drives every unit's
paths to one sign (the sign with which its feature was built), flipping the designed minority paths or starving them, and the sign of
a W2 edge is tied to the identity of the row it reads, so it flips exactly when the row re-targets. Paths that flip for other reasons
are the exception (one majority W3 edge).

In [ ]:
# Run 26 only: loads its four weight stacks (every 10 epochs) through the light loader
from layer_lenses.relu_analysis import edge_sign_flips, path_products, path_sign_summary, unit_path_sign_consensus
from layer_lenses.designed_init import probe_deviation_lineage
KEYS = ("hidden_layers.0.weight", "hidden_layers.1.weight", "hidden_layers.2.weight", "output_layer.weight")
r26 = load_weight_stacks(26, keys=KEYS); ep26 = r26["epochs"]
W1, W2, W3 = (r26["W"][k] for k in KEYS[:3]); Wo = r26["W"][KEYS[3]][:, 0, :]
unit_sign = np.array([1, -1] * 15)                                   # the units' net path sign at init (+-10 quanta, alternating)
ps = path_sign_summary(Wo, W3, W2, ep26, significant_ratio=0.1, tail=200)
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in ps.items() if not isinstance(v, np.ndarray)})
f2, f3, fo = edge_sign_flips(W2, ep26), edge_sign_flips(W3, ep26), edge_sign_flips(Wo, ep26)
minority = (W3[0] != 0) & (np.sign(Wo[0])[:, None] * np.sign(W3[0]) != unit_sign[None, :])
print(f"output weights flipped: {fo['net_flipped'].sum()} / 40; W3 edges flipped {f3['net_flipped'].sum()} / {(W3[0] != 0).sum()} "
      f"(minority {(f3['net_flipped'] & minority).sum()} / {minority.sum()}); W2 edges flipped {f2['net_flipped'].sum()} / {(W2[0] != 0).sum()} "
      f"(probe {f2['net_flipped'][:, :30].sum()} / 30, cue {f2['net_flipped'][:, 30:62].sum()} / {(W2[0][:, 30:62] != 0).sum()})")
cons = pd.DataFrame({"unit": range(30), "level": [max(0, int(np.log2(j // 2 + 1))) for j in range(30)], "consensus_init": unit_path_sign_consensus(Wo[0], W3[0], unit_sign),
                     "consensus_end": unit_path_sign_consensus(Wo[-1], W3[-1], unit_sign), "minority_flipped": (f3["net_flipped"] & minority).sum(0),
                     "born_w3_on_net_sign": ((W3[0] == 0) & (np.sign(Wo[-1])[:, None] * np.sign(W3[-1]) == unit_sign[None, :])).sum(0)})
print(cons.round(3).to_string(index=False))
# every designed W2 / W3 edge that ends with the opposite sign, with where its row sits at the end (W2) or whether it was a minority path (W3)
lin = probe_deviation_lineage(W1[-1], tree); C_end = (W1[-1] / np.linalg.norm(W1[-1], axis=1, keepdims=True)) @ tree.w_list.T
rows = [{"layer": "W2", "unit": j, "row": i, "row_kind": "probe" if i < 30 else "cue", "init": W2[0, j, i], "end": W2[-1, j, i], "first_flip_epoch": f2["first_flip_epoch"][j, i],
         "row_end_node": int(np.abs(C_end[i]).argmax()), "row_end_cos": C_end[i][np.abs(C_end[i]).argmax()]} for j, i in np.argwhere(f2["net_flipped"])]
rows += [{"layer": "W3", "unit": j, "row": k, "row_kind": "minority" if minority[k, j] else "majority", "init": W3[0, k, j], "end": W3[-1, k, j], "first_flip_epoch": f3["first_flip_epoch"][k, j]}
         for k, j in np.argwhere(f3["net_flipped"])]
flips26 = pd.DataFrame(rows); flips26.to_csv(RESULTS / "path_sign_flips_run26.csv", index=False)
print(flips26.groupby(["layer", "row_kind"]).agg(n=("unit", "size"), ratio_end_init_median=("end", lambda s: float(np.median(np.abs(s) / np.abs(flips26.loc[s.index, "init"]))))).round(2))

In [ ]:
# Run 26 only: when the designed edges flip (cumulative first flips by class) and the per-unit path-sign consensus
loss26 = pd.read_csv(RESULTS / "hierarchy_replayable_cuev" / "per_epoch_linear_free_kaiming_cosine_cuev_80_seed0.csv")
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for name, first in [("W3 minority", f3["first_flip_epoch"][minority]), ("W2 cue", f2["first_flip_epoch"][:, 30:62]), ("W2 probe", f2["first_flip_epoch"][:, :30])]:
    fe = np.sort(first[first >= 0]); ax[0].step(fe, np.arange(1, len(fe) + 1), where="post", label=f"{name} ({len(fe)})")
ax[0].set_xscale("log"); ax[0].set_xlabel("epoch of first sign change"); ax[0].set_ylabel("edges (cumulative)"); ax[0].legend(loc="upper left")
ax0 = ax[0].twinx(); ax0.plot(loss26.epoch, loss26.loss, color="k", lw=0.6, alpha=0.5); ax0.set_yscale("log"); ax0.set_ylabel("epoch loss")
ax[0].set_title("run 26: designed edges that change sign")
ax[1].bar(cons.unit - 0.2, cons.consensus_init, 0.4, label="init"); ax[1].bar(cons.unit + 0.2, cons.consensus_end, 0.4, label="end")
ax[1].set_ylim(0.5, 1.02); ax[1].set_xlabel("unit"); ax[1].set_ylabel("fraction of |w_k W3[k, j]| on the unit's net sign"); ax[1].legend(loc="lower left"); ax[1].set_title("path-sign consensus per unit")
plt.tight_layout(); plt.show()